In [16]:
"""Quick data profiling — prints structural overview of dataset.csv"""

import pandas as pd
import sys

DATA = "../dataset.csv"
SAMPLE_N = 500_000  # sample for heavy ops

In [17]:
df = pd.read_csv(DATA, parse_dates=["timestamp"])
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} cols\n")

# --- dtypes ---
print("=" * 60)
print("DTYPES")
print("=" * 60)
print(df.dtypes)

Shape: 5,000,000 rows x 15 cols

DTYPES
transaction_id                       str
account_id                           str
customer_id                          str
timestamp                 datetime64[us]
amount                           float64
balance_before_ngn               float64
balance_after_ngn                float64
transaction_type                     str
channel                              str
merchant_category_code           float64
merchant_name                        str
location_lga                         str
location_state                       str
device_id                            str
status                               str
dtype: object


In [ ]:
# --- nulls ---
print("\n" + "=" * 60)
print("NULL COUNTS & PERCENTAGES")
print("=" * 60)
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
null_df = pd.DataFrame({"nulls": nulls, "pct": null_pct})
print(null_df[null_df["nulls"] > 0])
if null_df["nulls"].sum() == 0:
    print("No nulls found.")


NULL COUNTS & PERCENTAGES
                          nulls    pct
merchant_category_code  2998137  59.96
merchant_name           2998137  59.96
device_id               2750464  55.01


In [8]:
# --- unique counts ---
print("\n" + "=" * 60)
print("UNIQUE VALUE COUNTS")
print("=" * 60)
for col in df.columns:
    print(f"  {col}: {df[col].nunique():,}")


UNIQUE VALUE COUNTS
  transaction_id: 5,000,000
  account_id: 100,000
  customer_id: 100,000
  timestamp: 848,014
  amount: 27,400
  balance_before_ngn: 5,000,000
  balance_after_ngn: 4,653,268
  transaction_type: 2
  channel: 7
  merchant_category_code: 14
  merchant_name: 45
  location_lga: 37
  location_state: 37
  device_id: 2,249,536
  status: 4


In [18]:
# --- numeric stats ---
print("\n" + "=" * 60)
print("NUMERIC STATS")
print("=" * 60)
num_cols = df.select_dtypes(include="number").columns.tolist()
print(df[num_cols].describe().round(2))


NUMERIC STATS
           amount  balance_before_ngn  balance_after_ngn  \
count  5000000.00          5000000.00         5000000.00   
mean     65621.99           336498.80          334980.25   
std     230311.87           452762.94          473451.41   
min        100.00             4577.12               0.00   
25%       4000.00            86098.43           73019.20   
50%      13300.00           187949.88          183106.65   
75%      45000.00           402859.47          408644.07   
max    5000000.00         14879436.33        14820636.33   

       merchant_category_code  
count              2001863.00  
mean                  5410.63  
std                    507.88  
min                   4814.00  
25%                   4899.00  
50%                   5411.00  
75%                   5812.00  
max                   7523.00  


In [10]:
# --- categorical value counts (top 10) ---
cat_cols = ["transaction_type", "channel", "status", "location_state", "merchant_category_code"]
print("\n" + "=" * 60)
print("CATEGORICAL VALUE COUNTS (top 10)")
print("=" * 60)
for col in cat_cols:
    if col not in df.columns:
        continue
    print(f"\n--- {col} ---")
    vc = df[col].value_counts(dropna=False).head(10)
    for val, cnt in vc.items():
        print(f"  {val}: {cnt:,} ({cnt/len(df)*100:.1f}%)")


CATEGORICAL VALUE COUNTS (top 10)

--- transaction_type ---
  debit: 3,250,541 (65.0%)
  credit: 1,749,459 (35.0%)

--- channel ---
  mobile: 1,749,043 (35.0%)
  pos: 1,501,370 (30.0%)
  atm: 999,469 (20.0%)
  web: 500,493 (10.0%)
  branch: 149,372 (3.0%)
  ussd: 75,336 (1.5%)
  agent: 24,917 (0.5%)

--- status ---
  success: 4,599,963 (92.0%)
  failed: 349,765 (7.0%)
  pending: 40,383 (0.8%)
  reversed: 9,889 (0.2%)

--- location_state ---
  Lagos: 1,225,836 (24.5%)
  Abuja (FCT): 542,759 (10.9%)
  Rivers: 446,381 (8.9%)
  Kano: 288,046 (5.8%)
  Anambra: 239,518 (4.8%)
  Oyo: 238,988 (4.8%)
  Imo: 186,312 (3.7%)
  Kaduna: 159,387 (3.2%)
  Delta: 135,097 (2.7%)
  Edo: 125,496 (2.5%)

--- merchant_category_code ---
  nan: 2,998,137 (60.0%)
  4814.0: 400,590 (8.0%)
  5411.0: 300,383 (6.0%)
  5812.0: 239,530 (4.8%)
  5541.0: 200,467 (4.0%)
  4900.0: 160,888 (3.2%)
  4899.0: 120,426 (2.4%)
  5999.0: 100,015 (2.0%)
  5814.0: 99,768 (2.0%)
  5651.0: 99,755 (2.0%)


In [11]:
# --- duplicates ---
print("\n" + "=" * 60)
print("DUPLICATE CHECK")
print("=" * 60)
dup_txn = df["transaction_id"].duplicated().sum()
print(f"Duplicate transaction_ids: {dup_txn:,}")
full_dups = df.duplicated().sum()
print(f"Fully duplicate rows: {full_dups:,}")


DUPLICATE CHECK
Duplicate transaction_ids: 0
Fully duplicate rows: 0


In [12]:
# --- timestamp range ---
print("\n" + "=" * 60)
print("TIMESTAMP RANGE")
print("=" * 60)
print(f"Min: {df['timestamp'].min()}")
print(f"Max: {df['timestamp'].max()}")


TIMESTAMP RANGE
Min: 2023-01-01 00:07:00
Max: 2024-12-30 23:59:00


In [14]:
# --- balance consistency check (sample) ---
print("\n" + "=" * 60)
print(f"BALANCE CONSISTENCY CHECK (sample of {SAMPLE_N:,})")
print("=" * 60)
sample = df.sample(n=min(SAMPLE_N, len(df)), random_state=42)
debit = sample[sample["transaction_type"] == "debit"]
credit = sample[sample["transaction_type"] == "credit"]

debit_ok = ((debit["balance_before_ngn"] - debit["amount"] - debit["balance_after_ngn"]).abs() < 0.01).mean()
credit_ok = ((credit["balance_before_ngn"] + credit["amount"] - credit["balance_after_ngn"]).abs() < 0.01).mean()
print(f"Debit balance consistent: {debit_ok*100:.2f}%")
print(f"Credit balance consistent: {credit_ok*100:.2f}%")


BALANCE CONSISTENCY CHECK (sample of 500,000)
Debit balance consistent: 89.34%
Credit balance consistent: 100.00%


In [ ]:
# --- negative amounts / balances ---
print("\n" + "=" * 60)
print("ANOMALY CHECK")
print("=" * 60)
print(f"Negative amounts: {(df['amount'] < 0).sum():,}")
print(f"Negative balance_before: {(df['balance_before_ngn'] < 0).sum():,}")
print(f"Negative balance_after: {(df['balance_after_ngn'] < 0).sum():,}")
print(f"Zero amounts: {(df['amount'] == 0).sum():,}")

print("\n" + "=" * 60)
print("SAMPLE ROWS")
print("=" * 60)
print(df.head(3))


ANOMALY CHECK
Negative amounts: 0
Negative balance_before: 0
Negative balance_after: 0
Zero amounts: 0

SAMPLE ROWS
                         transaction_id    account_id   customer_id  \
0  42ae7cb0-052b-4c28-a149-8d7d2cc558a1  ACC-00000334  CUS-00000334   
1  ce38456c-5bbb-4a83-992a-09e8fd59b49a  ACC-00004849  CUS-00004849   
2  9e69c19c-93c8-48fc-b2b8-e89d83b40936  ACC-00082189  CUS-00082189   

            timestamp   amount  balance_before_ngn  balance_after_ngn  \
0 2024-09-14 14:59:00   2100.0       248561.187746      246461.187746   
1 2024-12-26 01:54:00  73400.0       224711.795501      151311.795501   
2 2024-06-15 13:08:00  22600.0       114278.718868      136878.718868   

  transaction_type channel  merchant_category_code      merchant_name  \
0            debit     web                  4814.0             Airtel   
1            debit     web                  5411.0       Grand Square   
2           credit     pos                  7523.0  MegaPlaza Parking   

  location_l